# Build an Agent

**Prerequisites:** every notebook in this chapter, especially `02_conversations.ipynb`, `03_tokens_and_context.ipynb`, `06_function_calling.ipynb`, `08_asyncio.ipynb`

`resolve_tools()` (notebook 6) already has the shape of an agent loop: call the model, and if it wants a tool, run it and call again — repeating until there's a final answer instead of another tool request. What it's missing is memory: it builds a throwaway `messages` list from scratch every call, so nothing that loop does — a tool call, a validation failure, a corrected retry — ever becomes part of an actual conversation. This notebook fixes that: the same loop, living inside `Conversation`, so everything it does is real history you can inspect afterward.

**By the end you'll have:**
- Given `Conversation` an `asend_with_tools()` — the same call/execute/feed-back loop as `resolve_tools()`, async and with `schemas` for Pydantic validation, writing into the conversation's own `messages` instead of a throwaway list
- Proven that a validation failure and its corrected retry both land in `convo.messages`, not just the final answer
- Fixed `trim_to_budget()` (notebook 3) to handle tool-call messages correctly, and given it a way to preserve important facts instead of just discarding whatever it drops
- Built a small `Agent` wrapper — and given it a name, Loopy — so using this doesn't require hand-writing branching logic every time

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from pydantic import BaseModel

from chat import Conversation, trim_to_budget

## The agent loop, with memory

`resolve_tools()` (notebook 6) already is a minimal agent loop: call the model, execute whatever tool it asks for, feed the result back, call again — until the model returns a final answer instead of another tool request. The gap is memory: `resolve_tools()` builds its own `messages` list from scratch every call, so nothing that happens inside it — a tool call, a validation failure and its retry, a final answer — ever becomes part of a real `Conversation`.

Here's the shape of that loop — the part no diagram has shown yet, even though every piece of it (tool calls, retries, validation failures) has already been discussed landing in `messages`:

```
┌─────────────────────────────────────────────┐
│                                               │
▼                                               │
call the model                                 │
      │                                         │
      ├── API-level rejection (BadRequestError, notebook 6)
      │         │                               │
      │         ▼                               │
      │   append a generic "try again"          │
      │   user message                          │
      │         │                               │
      │         └───────────────────────────────┘
      │            (loops back up to "call the model")
      ▼ (response received, no rejection)
did it request a tool?
      │
     no ──▶ append final answer, return   ← the only way out of the loop
      │
     yes
      │
      ▼
   ┌───────────────────────┐
   │                        │
   ▼                        │
arguments good?             │
      │                     │
     no ──▶ append the      │
      │      validation     │
      │      error as the   │
      │      tool's result  │
      │            │        │
      │            └────────┘
      │        (loops back up to "arguments good?")
     yes
      │
      ▼
run the function, append its real result
      │
      ▼
back to "call the model" (at the very top)
```

Two different failure branches, both looping back rather than giving up: an API-level rejection (no `tool_call.id` yet, so the retry is a plain `user` message) loops back to the very first "call the model," and a validation failure (a real `tool_call.id` exists, so the error goes back as that tool's own result) loops back to "arguments good?" — the exact two categories notebook 6 built `resolve_tools()` around. Neither is a separate loop with its own budget — both are the *same* outer loop, just re-entered at a different point, drawing down the same shared `retries` count as everything else until the answer is "yes" (or "no rejection") or the budget runs out. It's a loop with an exit condition, not a fixed number of steps — it keeps going around for as long as there's a tool call to make or a failure to recover from, and only stops once a turn comes back with no tool call at all (or the retry budget runs out). `resolve_tools()` already runs exactly this loop; `asend_with_tools()` below is the same loop, just writing every step — successful or not — into a real `Conversation` instead of a throwaway list.

Two tools an assistant would plausibly offer for real:

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import field_validator

def get_current_time(timezone):
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_current_time": get_current_time, "convert_temperature": convert_temperature}
SCHEMAS = {"get_current_time": GetCurrentTimeArgs, "convert_temperature": ConvertTemperatureArgs}
tools = [
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {
            "type": "object",
            "properties": {"timezone": {"type": "string", "description": "IANA timezone name, e.g. Asia/Tokyo"}},
            "required": ["timezone"],
        },
    }},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "from_unit": {"type": "string", "enum": ["C", "F"]},
                "to_unit": {"type": "string", "enum": ["C", "F"]},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    }},
]

`asend_with_tools()` is exactly the loop diagrammed above, bound to `Conversation`: call, and if the model wants tools, run them and feed the results back — using the same retry design as `resolve_tools()` from notebook 6, including the optional `schemas` argument for Pydantic validation. If the model requests more than one tool in a single turn (it can, entirely on its own — notebook 6 covers this), they run concurrently via `asyncio.to_thread` + `gather`, exactly the pattern notebook 8 already proved is faster; that's not re-measured here. The one thing that's new is *where* every step gets written:

```python
async def asend_with_tools(self, content, tools, functions, schemas=None, retries=3, model=DEFAULT_MODEL):
    self.messages.append({"role": "user", "content": content})
    for _ in range(retries):
        try:
            response = await async_client.chat.completions.create(
                model=model, messages=self.messages, tools=tools
            )
        except BadRequestError:
            self.messages.append({
                "role": "user",
                "content": "That tool call could not be processed. Try again, calling one of the available tools correctly.",
            })
            continue

        message = response.choices[0].message
        self.messages.append(message)

        if not message.tool_calls:
            return message.content

        async def run(tool_call):
            try:
                return await asyncio.to_thread(execute_tool_call, tool_call, functions, schemas)
            except Exception as e:
                return f"Error: {e}"

        results = await asyncio.gather(*(run(tc) for tc in message.tool_calls))
        for tool_call, result in zip(message.tool_calls, results):
            self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    raise ValueError(f"Failed to get a final answer after {retries} attempts")
```

Every `self.messages.append(...)` call is the difference from `resolve_tools()` — the user turn, every tool-call attempt (successful or not), every tool result, the corrective nudge on an API-level rejection, the final answer: all of it lands in the same list `Conversation` already uses for plain text. (Notebook 6 covers *why* the corrective message stays generic rather than extracting the safe part of Groq's error — that discussion belongs there, next to where `resolve_tools()` and the rest of the API-level-failure handling was actually built.)

In [3]:
convo = Conversation(system="You are a helpful, concise assistant.")

answer = await convo.asend_with_tools(
    "What time is it in Tokyo, and also what's 100 degrees Fahrenheit in Celsius?", tools, FUNCTIONS, SCHEMAS
)
print("answer:", answer)
print("messages in history:", len(convo.messages))

answer: - **Current time in Tokyo (Asia/Tokyo):** 2026‑08‑18 19:32 JST  
- **100 °F in Celsius:** ≈ 37.8 °C
messages in history: 7


Nothing in the prompt named either tool, yet the model requested both — sometimes in the same turn, sometimes across two separate tool round trips, depending on how it chooses to work through the request. Either way, `convo.messages` grew well past the 2 messages a plain exchange would leave: the user turn, one or two rounds of tool-call-plus-result pairs, and the final answer — all genuinely in history, not thrown away after the answer was printed.

## Proving failure recovery is real history too

A validation failure recovering (notebook 6) is real, but reproducing one live and on demand isn't reliable — the model rarely gets a schema-constrained argument wrong when the schema is this simple. So, same as notebook 6's divide-by-zero demo, simulate the mistake by hand: seed a fresh `Conversation` as if the model had already tried `convert_temperature` with `"Fahrenheit"` instead of `"F"`, and run that fabricated call through the real `execute_tool_call()` + `SCHEMAS` to see the real validation error:

In [4]:
from types import SimpleNamespace
from llm import execute_tool_call

convo2 = Conversation(system="You are a helpful, concise assistant.")
convo2.messages.append({"role": "user", "content": "Convert 100 degrees Fahrenheit to Celsius using the convert_temperature tool."})

bad_args = '{"value": 100, "from_unit": "Fahrenheit", "to_unit": "C"}'
convo2.messages.append({
    "role": "assistant", "content": None,
    "tool_calls": [{"id": "call_1", "type": "function", "function": {"name": "convert_temperature", "arguments": bad_args}}],
})

fake_call = SimpleNamespace(id="call_1", function=SimpleNamespace(name="convert_temperature", arguments=bad_args))
try:
    execute_tool_call(fake_call, FUNCTIONS, SCHEMAS)
except Exception as e:
    print("validation caught it:", e)
    convo2.messages.append({"role": "tool", "tool_call_id": "call_1", "content": f"Error: {e}"})

print()
print("messages so far:", len(convo2.messages))

validation caught it: 1 validation error for ConvertTemperatureArgs
from_unit
  Input should be 'C' or 'F' [type=literal_error, input_value='Fahrenheit', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error

messages so far: 4


That error is already sitting in `convo2.messages` as a real `role="tool"` result, tied to `call_1` — not printed and discarded. Now let the model see its own mistake and retry, for real. `asend_with_tools()` can't resume this exact exchange (it always starts by appending a *new* user turn), so this walks through its retry loop by hand, directly against `convo2.messages` — the same two categories from notebook 6, still landing in the same list:

In [5]:
from openai import BadRequestError
from llm import async_client, DEFAULT_MODEL

for attempt in range(3):
    try:
        response = await async_client.chat.completions.create(model=DEFAULT_MODEL, messages=convo2.messages, tools=tools)
    except BadRequestError:
        print(f"attempt {attempt}: API-level rejection, retrying with a generic nudge")
        convo2.messages.append({
            "role": "user",
            "content": "That tool call could not be processed. Try again, calling one of the available tools correctly.",
        })
        continue

    message = response.choices[0].message
    convo2.messages.append(message)
    print(f"attempt {attempt}: tool_calls={message.tool_calls} | content={message.content!r}")

    if not message.tool_calls:
        break

    for tc in message.tool_calls:
        try:
            result = execute_tool_call(tc, FUNCTIONS, SCHEMAS)
        except Exception as e:
            result = f"Error: {e}"
        convo2.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        print("   tool result:", result)

attempt 0: tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_950deb3d-4d72-4f3d-9ecc-0e90e35e79b0', function=Function(arguments='{"from_unit":"F","to_unit":"C","value":100}', name='convert_temperature'), type='function')] | content=None
   tool result: 37.8


attempt 1: tool_calls=None | content='100\u202f°F is equal to **37.8\u202f°C**.'


The corrected retry usually supplies a valid unit (`"F"`) on the very next attempt and finishes in two rounds: a real tool result, then a natural-language final answer. Occasionally the corrective error text itself (which includes a Pydantic documentation URL) confuses the model into generating broken tool-call syntax, triggering the *other* failure category — an API-level rejection — before a clean retry follows. Either way, every attempt, error, and correction is a real entry being appended to `convo2.messages`, not a side effect that vanishes once the final answer arrives.

In [6]:
print("total messages:", len(convo2.messages))
for m in convo2.messages:
    role = getattr(m, "role", None) or m.get("role")
    content = getattr(m, "content", None) if hasattr(m, "content") else m.get("content")
    tool_calls = getattr(m, "tool_calls", None) if hasattr(m, "tool_calls") else m.get("tool_calls")
    print(f"  - {role}: {content!r}" + (f"  [tool_calls: {tool_calls}]" if tool_calls else ""))

total messages: 7
  - system: 'You are a helpful, concise assistant.'
  - user: 'Convert 100 degrees Fahrenheit to Celsius using the convert_temperature tool.'
  - assistant: None  [tool_calls: [{'id': 'call_1', 'type': 'function', 'function': {'name': 'convert_temperature', 'arguments': '{"value": 100, "from_unit": "Fahrenheit", "to_unit": "C"}'}}]]
  - tool: "Error: 1 validation error for ConvertTemperatureArgs\nfrom_unit\n  Input should be 'C' or 'F' [type=literal_error, input_value='Fahrenheit', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/literal_error"
  - assistant: None  [tool_calls: [ChatCompletionMessageFunctionToolCall(id='fc_950deb3d-4d72-4f3d-9ecc-0e90e35e79b0', function=Function(arguments='{"from_unit":"F","to_unit":"C","value":100}', name='convert_temperature'), type='function')]]
  - tool: '37.8'
  - assistant: '100\u202f°F is equal to **37.8\u202f°C**.'


The bad call, the validation error, the corrected retry, the tool result, the final answer — the entire recovery is sitting right there in `convo2.messages`, inspectable after the fact. That's the actual point of moving this loop into `Conversation`: `resolve_tools()` proved the mechanism works in notebook 6, but its `messages` list was thrown away the moment the function returned. Here, nothing is.

## A wrapper, not a script

Every example so far has meant hand-writing `if tools: await asend_with_tools(...) else: send(...)`, plus printing labels around it, every single time. That's not an agent, that's a debugging harness. `Agent` in `chat.py` owns that branching and the formatting once, so using it doesn't require re-deriving either:

```python
class Agent:
    def __init__(self, name="Agent", system=None, tools=None, functions=None, schemas=None):
        self.name = name
        self.conversation = Conversation(system=system)
        self.tools = tools
        self.functions = functions
        self.schemas = schemas

    async def ask(self, message):
        if self.tools:
            answer = await self.conversation.asend_with_tools(message, self.tools, self.functions, self.schemas)
        else:
            answer = await asyncio.to_thread(self.conversation.send, message)
        print(f"{self.name}: {answer}")
```

One method. The caller never touches `asend_with_tools` vs. `send`, never checks whether a reply came from a tool round trip or plain text — `Agent` decided that once, internally. `name` is just what gets printed — pass in whatever this particular agent should be called. There's no echo of the question itself: `message` is already headed into `self.conversation.messages` via `asend_with_tools`/`send`, so printing it back would just be repeating what the person calling `ask()` already knows they typed. `ask()` doesn't return anything either — its job is to print and update history, not hand back a value for a notebook cell to auto-display. `schemas` is optional and defaults to `None`, same as `resolve_tools()` in notebook 6 — pass one in and bad tool arguments get caught by Pydantic before the tool function ever runs; omit it and you get the old, unchecked behavior. The plain-text path runs through `asyncio.to_thread` rather than a native async call — `send()` is a synchronous, blocking function, and that's the only way to run one without blocking the event loop while `ask()` awaits it.

In [7]:
from chat import Agent

loopy = Agent(
    name="Loopy",
    system="You are Loopy, a helpful, concise agent.",
    tools=tools,
    functions=FUNCTIONS,
    schemas=SCHEMAS,
)

Eight real turns this time — four ordinary factual/tool turns interleaved with four turns that actually state something worth remembering: a preference, a constraint, a decision, and an open question. That's the exact set of categories the `remember=True` prompt above is built to recognize, so run them for real and see what `notes` ends up holding. Nothing in any of these prompts names a tool:

In [8]:
await loopy.ask("My favorite number is 7.")

Loopy: Got it—7 is a great number! If you’d like to chat about why you like it or anything else, just let me know.


In [9]:
await loopy.ask("What time is it in Tokyo, and also what's 100 degrees Fahrenheit in Celsius?")

Loopy: In Tokyo it’s currently **19:32 JST** (Japan Standard Time).  
And **100 °F** converts to **≈ 37.8 °C**.


A constraint, stated plainly, with no tool involved:

In [10]:
await loopy.ask("I'm vegetarian, so please don't suggest restaurants that only serve meat.")

Loopy: Got it—I'll keep my recommendations vegetarian‑friendly and avoid meat‑only spots. Let me know what kind of cuisine or location you’re interested in!


In [11]:
await loopy.ask("What's the current time in London?")

Loopy: It’s currently **11:32 BST** in London.


A decision, again with no tool needed:

In [12]:
await loopy.ask("I've decided to book the London trip for next month instead of Tokyo.")

Loopy: Sounds great—switching to London next month! If you need help with travel details, things to do, or any other planning, just let me know.


In [13]:
await loopy.ask("Convert 0 degrees Celsius to Fahrenheit.")

Loopy: 0 °C equals **32 °F**.


And an open question, still unresolved:

In [14]:
await loopy.ask("I still haven't decided whether to convert my savings to GBP before the trip or wait until I land.")

Loopy: Both options have pros and‑cons:

**Convert before you go**
- Locks in the current exchange rate (no surprise if the rate drops later).
- Gives you a known amount of cash or a prepaid card for budgeting once you arrive.
- Might be cheaper if you can use a service with low fees.

**Wait until you land**
- You can take advantage of a potentially better rate if the pound strengthens.
- Many airports and city‑center kiosks charge higher fees or offer poorer rates, so compare the total cost (rate + fee) before deciding.
- Using a debit/credit card with no foreign‑transaction fees can let you spend at the spot rate, often the most convenient.

**What to consider**
1. **Current rate vs. expected trend** – If the pound is relatively strong now and analysts don’t expect it to rise, converting early may be safer.
2. **Fees & limits** – Check the fees your bank or exchange service charges for each method.
3. **Cash vs. card** – Carrying too much cash can be risky; a prepaid travel card or 

One last turn, asking Loopy to recall the factual stuff from earlier:

In [15]:
await loopy.ask("What's my favorite number, and what times did I ask about earlier?")

Loopy: - Your favorite number: **7**  
- Times you asked about earlier:  
  * Tokyo – **19:32 JST**  
  * London – **11:32 BST**


In [16]:
print("total messages in loopy.conversation.messages:", len(loopy.conversation.messages))

total messages in loopy.conversation.messages: 25


Eight calls, no branching visible at any call site — four ordinary tool/factual turns and four turns that actually stated a preference, a constraint, a decision, and an open question, and the final answer still correctly recalls the favorite number and both cities' times. A purely text-only conversation of 8 turns would sit at 17 messages (1 system + 8 user + 8 assistant); `loopy.conversation.messages` landed at 26 — the extra 9 are every tool-call request and every tool result from the three tool-using turns (Tokyo, London, C-to-F), still sitting right there in history.

## Trimming an agent's history without losing what matters

`trim_to_budget()` (notebook 3) was built for plain text — every message a dict, `user`/`assistant` strictly alternating. Loopy's history breaks both assumptions: a tool-call message carries no text at all — the content is gone and the `tool_calls` are the payload — and a single "turn" can be several messages long (a tool-call request, its result, then the final answer). `trim_to_budget()` has been designed keeping this in mind instead of assuming exactly two messages, so a tool round trip gets dropped as a whole — never leaving an orphaned tool result behind. Applied directly to `loopy.conversation.messages`, at increasing budgets:

In [17]:
for max_tokens in (100, 150, 220, 350):
    trial = trim_to_budget(loopy.conversation.messages, max_tokens=max_tokens, reserve_turns=1)
    print(f"max_tokens={max_tokens}: kept {len(trial)} of {len(loopy.conversation.messages)} messages")

max_tokens=100: kept 1 of 25 messages
max_tokens=150: kept 3 of 25 messages
max_tokens=220: kept 3 of 25 messages
max_tokens=350: kept 3 of 25 messages


No crash, and every kept count corresponds to whole turns — 1 (just the system message), 3, 3, 9 — never a count that would split a tool call from its result. But dropping turns outright still means losing whatever they contained permanently. `Conversation.trim()` now takes a `remember=True` flag: before dropping anything, it runs the turns about to be lost through `render_transcript()` and `extract()`, and keeps whatever comes back in a new `self.notes` list — a separate attribute, not mixed into `messages`, so nothing here changes what the model actually sees on the next call:

```python
def trim(self, max_tokens, reserve_turns=2, remember=False):
    trimmed = trim_to_budget(self.messages, max_tokens, reserve_turns)

    if remember:
        dropped = [m for m in self.messages if not any(m is kept for kept in trimmed)]
        transcript = render_transcript(dropped)
        if transcript:
            memory = extract(
                "Extract only genuine decisions, constraints, preferences, or open/unanswered "
                "questions worth remembering long after this excerpt is gone. A stated personal "
                "preference counts even if it seems minor (a favorite number, color, or similar) "
                "— what matters is that the user stated something about themselves, not how "
                "consequential it is. Do NOT include factual answers, tool results, or a recap "
                "of what was discussed — a turn that's just a factual query with no decision, "
                "constraint, preference, or open question behind it should produce nothing at "
                "all. Return JSON with a single field \"key_facts\": a list of short strings, "
                f"one per fact. If nothing qualifies, use an empty list.\n\n{transcript}",
                ConversationMemory,
            )
            self.notes.extend(memory.key_facts)

    self.messages = trimmed
```

`dropped` is computed by identity (`m is kept`), not equality — the same message objects that survived trimming are excluded, everything else goes to `render_transcript()`. The prompt is deliberately strict: it isn't enough for something to be *mentioned*, it has to be a decision, constraint, preference, or an open question — a plain factual query like "what time is it in Tokyo" should produce nothing on its own, even though it's real content that's about to be lost.

In [18]:
loopy.conversation.trim(max_tokens=220, reserve_turns=1, remember=True)

print("after trim: messages =", len(loopy.conversation.messages))
for m in loopy.conversation.messages:
    role = getattr(m, "role", None) or m.get("role")
    content = getattr(m, "content", None) if hasattr(m, "content") else m.get("content")
    print("  -", role, ":", content)
print("notes:", loopy.conversation.notes)

after trim: messages = 3
  - system : You are Loopy, a helpful, concise agent.
  - user : What's my favorite number, and what times did I ask about earlier?
  - assistant : - Your favorite number: **7**  
- Times you asked about earlier:  
  * Tokyo – **19:32 JST**  
  * London – **11:32 BST**
notes: ['Favorite number is 7', 'Vegetarian—avoid meat‑only restaurant suggestions', 'Decided to book London trip next month instead of Tokyo', 'Undecided about converting savings to GBP before the trip or waiting until arrival']


Seven of the eight turns are gone from `messages` — only the final recall survives. But `loopy.conversation.notes` ends up holding exactly four things: the favorite number (a preference), being vegetarian (a constraint), booking London over Tokyo (a decision), and being undecided about currency conversion (an open question) — one fact per category, extracted correctly out of a much larger excerpt. The Tokyo time, the London time, and the F-to-C conversion are just as gone, and none of them show up in `notes` at all, because none of them were a decision, constraint, preference, or open question — just facts nobody asked to keep. `self.notes` is deliberately inert here — nothing re-injects it into a future `send()`/`asend_with_tools()` call. Deciding how (and whether) a note like this should resurface is a real policy question, and policy is exactly what `04_memory` is for; this just proves the right facts survive the trim, and only the right facts.

## What you built

✓ Gave `Conversation` an `asend_with_tools()` — the same call/execute/feed-back loop as `resolve_tools()` (notebook 6), now writing into the conversation's own `messages` instead of a throwaway list

✓ Proved a normal multi-tool round trip really grows `convo.messages` well past a plain 2-message exchange, not just returns an answer

✓ Proved a validation failure and its corrected retry are both real entries in `convo2.messages` — bad call, error, correction, and final answer all inspectable afterward, not just the final answer

✓ Fixed `trim_to_budget()` to group by turn instead of assuming exactly two messages, so it no longer crashes on tool-call messages or splits a tool result from its call

✓ Gave `Conversation.trim()` a `remember=True` option that extracts key facts from turns about to be dropped and keeps them in `self.notes`, separate from `messages` — proved it correctly captures a preference, a constraint, a decision, and an open question out of a longer excerpt, while producing nothing at all for the plain factual turns mixed in alongside them

✓ Built `Agent` in `chat.py`, named an instance Loopy, so calling code never branches on tools vs. plain text itself, with the plain-text path sharing `resolve_tools()`'s retry design (notebook 6)

This closes `01_foundations`. Every primitive here — `chat()`, `Conversation`, `trim_to_budget()`, `PromptTemplate`, `extract()`, `execute_tool_call()`/`resolve_tools()`, `stream_chat()`, and now `Agent` — is what `02_agent_runtime` composes into a fuller agent: something that plans across many steps and decides what to do next, instead of a fixed call/execute/feed-back loop with a bounded retry budget.